# 🧪 W8-D5 概念实验：Capability 与 Connector

> 配套阅读：`第8周-Day5-Capability与Connector.md`（ADR-001 分类法、控制面/执行面、Gap Analysis 在那边）
>
> 四个可运行实验对应 md 的四条主线：
> 1. **Capability 不可变性**：ID 语法校验 + published 后 frozen（对应 catalog.py 的设计）
> 2. **Capability × Industry 正交模型**（ADR-003）：矩阵复用 vs 专用集成的数量账
> 3. **Capability Resolution**（ADR-004 §6.3）：受治理解析 + fail-closed
> 4. **Connector 治理缺失**（最大 Gap）：用熔断器状态机演示治理层该长的样子

环境：仅 numpy / 标准库 / matplotlib，无网络。

## 实验 1：Capability 目录 —— 校验、注册、发布后不可变

Capability 是平台自持的"专科能力"（区别于外包的 Plugin）。两个关键约束：
- **ID 语法**：全小写点分（`knowledge.query`），机械可校验
- **发布即冻结**：draft 可改；published 后核心字段不可修改——防止"已上线能力悄悄变形"

In [ ]:
import re
from dataclasses import dataclass, field, replace as dc_replace

CAP_ID = re.compile(r"^[a-z0-9]+(\.[a-z0-9]+)*$")

def make_capability(cap_id, version, inputs, outputs, effect="read_only"):
    if not CAP_ID.match(cap_id):
        raise ValueError(f"非法 capability_id: {cap_id}（须为小写点分式）")
    return dict(capability_id=cap_id, version=version, status="draft",
                input_schema=inputs, output_schema=outputs, effect_policy=effect)

class Catalog:                          # catalog.py 的极简版
    def __init__(self): self.items = {}
    def register(self, cap): self.items[cap["capability_id"]] = cap
    def publish(self, cap_id):
        cap = self.items[cap_id]
        self.items[cap_id] = {**cap, "status": "published",
                              "_frozen": True}          # 发布 → 冻结核心字段
    def patch(self, cap_id, **changes):
        cap = self.items[cap_id]
        if cap.get("_frozen"):
            core = set(changes) & {"capability_id", "version", "input_schema",
                                   "output_schema", "effect_policy"}
            if core:
                raise PermissionError(f"已发布能力不可修改核心字段: {core}")
        self.items[cap_id] = {**cap, **changes}

catalog = Catalog()
catalog.register(make_capability("knowledge.query", "1.0.0",
                  inputs={"question": "string"}, outputs={"answer": "string"}))
catalog.register(make_capability("workflow.execute", "1.0.0",
                  inputs={"workflow_id": "string"}, outputs={"run_id": "string"}))
try:
    catalog.register(make_capability("Knowledge-Query", "1.0.0",
                      inputs={}, outputs={}))
except ValueError as e:
    print(f"✓ ID 语法校验拦截：{e}")

catalog.patch("knowledge.query", description="知识查询")   # draft 阶段随便补充说明
catalog.publish("knowledge.query")
try:
    catalog.patch("knowledge.query", effect_policy="conditional_write")
except PermissionError as e:
    print(f"✓ 发布后改核心字段被拒：{e}")
catalog.patch("knowledge.query", display_name="知识查询")   # 非核心字段可改
print("✓ 非核心字段（display_name）可更新：", catalog.items["knowledge.query"]["display_name"])
print("目录:", {k: v["status"] for k, v in catalog.items.items()})

## 实验 2：Capability × Industry 正交模型（ADR-003）

Plugin 模式 = 每个行业写一个专用插件（M×N 集成）；
正交模型 = Capability 与 Industry 两个维度独立演化，组合时才相交。
用适用度矩阵算这笔账：矩阵里大部分格子是可复用的，只有少数需要行业定制。

In [ ]:
import numpy as np
rng = np.random.default_rng(8)

CAPS = ["knowledge.query", "workflow.execute", "doc.extract", "report.generate",
        "ticket.dispatch", "quality.eval", "data.validate", "notify.send",
        "identity.verify", "audit.trace"]
INDUSTRIES = ["制造", "零售", "医疗", "金融", "政务"]

# 模拟适用度矩阵：0=不适用 1=通用可用 2=需行业适配
M = rng.choice([0, 1, 2], size=(len(CAPS), len(INDUSTRIES)), p=[0.2, 0.6, 0.2])
usable = (M > 0)
print(f"{'Capability':<20}" + "".join(f"{i:>5}" for i in INDUSTRIES) + "   复用行业数")
for i, cap in enumerate(CAPS):
    print(f"{cap:<20}" + "".join(f"{('—' if M[i,j]==0 else '✓' if M[i,j]==1 else '△'):>5}"
          for j in range(len(INDUSTRIES))) + f"{usable[i].sum():>8}")
print("\n✓=通用可用  △=需行业适配  —=不适用")

n_plugin_integrations = usable.sum() * 2        # Plugin 模式：每个格子从零写
n_orthogonal = len(CAPS) + int((M == 2).sum())  # 正交模型：能力一份 + 适配层一份
print(f"\nPlugin 模式集成工作量（格数×2）   ≈ {n_plugin_integrations} 份")
print(f"正交模型（能力 {len(CAPS)} 份 + 适配 {int((M==2).sum())} 份）≈ {n_orthogonal} 份")
print(f"复用率 {(M==1).sum()}/{M.size} 的格子完全通用 → 新增一个行业只需写适配层，不重写能力。")

## 实验 3：Capability Resolution（ADR-004 §6.3）—— 受治理解析 + fail-closed

解析器只回答"这个 Capability 依赖由哪个 SkillRelease 实现"，它：
- **不执行业务、不代表授权、不碰 Connector**
- 打分选择：健康度 > 版本匹配 > 成本；**无可用实现时 fail-closed**（抛错，而不是随便挑一个）

In [ ]:
RELEASES = [   # 模拟已发布的 SkillRelease（声明它实现了哪个 capability）
    {"release": "kb-rag@1.4", "capability": "knowledge.query", "health": 0.99,
     "ver_match": 1.0, "cost": 0.6},
    {"release": "kb-hybrid@0.9", "capability": "knowledge.query", "health": 0.87,
     "ver_match": 0.8, "cost": 0.4},
    {"release": "wf-engine@2.1", "capability": "workflow.execute", "health": 0.98,
     "ver_match": 1.0, "cost": 0.5},
]

def resolve(capability_id, min_health=0.9):
    """受治理解析：返回最优 release；解析≠执行≠授权。"""
    cands = [r for r in RELEASES if r["capability"] == capability_id
             and r["health"] >= min_health]
    if not cands:
        raise LookupError(f"fail-closed：'{capability_id}' 无满足健康阈值的实现")
    return max(cands, key=lambda r: 0.5 * r["health"] + 0.3 * r["ver_match"]
               + 0.2 * (1 - r["cost"]))

for cap in ["knowledge.query", "workflow.execute", "ticket.dispatch"]:
    try:
        r = resolve(cap)
        print(f"✓ {cap:<20} → {r['release']}  (score={0.5*r['health']+0.3*r['ver_match']+0.2*(1-r['cost']):.2f})")
    except LookupError as e:
        print(f"✗ {cap:<20} → {e}")
print("\n解读：knowledge.query 有两个实现，按 健康50%+版本30%+成本20% 选优；")
print("ticket.dispatch 没有任何实现 → fail-closed，绝不允许'降级放行'。")
print("注意 resolve() 只做选择：真正执行还要走 SkillRelease 流程 + 权限校验（两层分离）。")

## 实验 4：Connector 治理 —— 熔断器状态机 + 全链路可视化

md 的最大 Gap：MCP Connector 嵌在 Workflow 里，无独立治理。演示治理层的基本件——
**熔断器**：CLOSED（正常）→ 失败率超阈 → OPEN（快速失败）→ 冷却后 HALF_OPEN 探测 →
成功回 CLOSED / 失败回 OPEN。用一段真实失败序列驱动状态机，画出状态与延迟。

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

class CircuitBreaker:
    def __init__(self, fail_threshold=0.5, window=8, cooldown=5):
        self.fail_threshold, self.window, self.cooldown = fail_threshold, window, cooldown
        self.state, self.history, self.opened_at = "CLOSED", [], -999
    def call(self, t, ok):
        if self.state == "OPEN":
            if t - self.opened_at >= self.cooldown:
                self.state = "HALF_OPEN"; self.history = []
            else:
                return "fast_fail"                      # 不打下游，立即失败
        self.history.append(0 if ok else 1)
        if len(self.history) >= self.window and self.state == "CLOSED":
            if np.mean(self.history[-self.window:]) > self.fail_threshold:
                self.state = "OPEN"; self.opened_at = t
        elif self.state == "HALF_OPEN":
            self.state = "CLOSED" if ok else "OPEN"; self.opened_at = t
        return "ok" if ok else "error"

rng = np.random.default_rng(19)
cb, states, latencies = CircuitBreaker(), [], []
outcomes = list(rng.random(40) < 0.65) + [False] * 6 + list(rng.random(15) < 0.9)
label = {"CLOSED": 0, "HALF_OPEN": 1, "OPEN": 2}
for t, ok in enumerate(outcomes):
    res = cb.call(t, ok)
    states.append(label[cb.state])
    latencies.append(800 if res == "fast_fail" else (900 + 100 * rng.random() if ok else 950))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
im = axes[0].imshow(M, aspect="auto", cmap="YlOrBr", vmin=0, vmax=2)
axes[0].set_xticks(range(len(INDUSTRIES))); axes[0].set_xticklabels(INDUSTRIES)
axes[0].set_yticks(range(len(CAPS))); axes[0].set_yticklabels(CAPS, fontsize=8)
axes[0].set_title("Capability × Industry 适用度（实验2矩阵）")
plt.colorbar(im, ax=axes[0], ticks=[0, 1, 2], label="0不适用/1通用/2需适配")

axes[1].step(range(len(states)), states, where="post", color="#d62828", lw=1.6)
axes[1].set_yticks([0, 1, 2]); axes[1].set_yticklabels(["CLOSED", "HALF_OPEN", "OPEN"])
axes[1].set_xlabel("请求序号"); axes[1].set_title("Connector 熔断器：连续失败 → OPEN → 冷却探测恢复")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

n_fast = sum(1 for s in states if s == 2)
print(f"本次序列触发 OPEN 期间 fast_fail {n_fast} 次：下游被保护，调用方拿到快速失败而非拖垮式超时。")
print("这正是 md 说的'独立 Connector 治理层'要补的基本能力之一（还有版本/效果策略/登记）。")

## 小结

- Capability = 平台自持能力：ID 可校验、published 后核心字段冻结
- 正交模型把 M×N 集成压成 M+N；适配层才是行业差异的归宿
- Resolution 只做"选择"，不做执行/授权；无实现时 fail-closed
- 最大 Gap 在 Connector：熔断/限流/登记/效果策略——治理层四件套